# Baseline Knowledge Evaluation — Qwen2.5-3B Instruct
Checks how much the model knows about 10 people from the RWKU dataset

In [2]:
import sys
sys.path.append('..')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils import load_rwku_datasets, evaluate_model, evaluate_neighbours

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: mps


In [3]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16)
model = model.to(DEVICE)
model.eval()
print("Model ready")

Loading model: Qwen/Qwen2.5-3B-Instruct


Fetching 2 files:   0%|          | 0/2 [13:22<?, ?it/s]
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
print("Loading RWKU datasets...")
forget_data, neighbor_data, train_data = load_rwku_datasets()

print(f"Forget questions:    {len(forget_data)}")
print(f"Neighbour questions: {len(neighbor_data)}")
print(f"Train passages:      {len(train_data)}")

In [ ]:
PEOPLE = [
    "50 Cent",
    "Taylor Swift",
    "Elon Musk",
    "Stephen King",
    "Beyoncé",
    "Kanye West",
    "Jay-Z",
    "Justin Bieber",
    "LeBron James",
    "Donald Trump",
]

## Forget dataset

In [ ]:
forget_results = {}

for person in PEOPLE:
    print(f"\n{'='*60}")
    print(f"Person: {person}")
    print('='*60)

    person_data = forget_data.filter(lambda x: person in x['subject'])
    questions = person_data['query']
    answers = person_data['answer']

    if len(questions) == 0:
        print(f"  No questions found for '{person}'.")
        continue

    accuracy = evaluate_model(model, tokenizer, questions, answers, DEVICE)
    forget_results[person] = {"accuracy": accuracy, "questions": len(questions)}

In [ ]:
print("=" * 50)
print(f"{'Person':<25} {'Accuracy':>10} {'Questions':>10}")
print("-" * 50)
for person, data in sorted(forget_results.items(), key=lambda x: -x[1]['accuracy']):
    print(f"{person:<25} {data['accuracy']:>10.2f}% {data['questions']:>10}")
print("=" * 50)
avg = sum(d['accuracy'] for d in forget_results.values()) / len(forget_results)
print(f"{'Average':<25} {avg:>10.2f}%")

## Neighbour dataset

In [ ]:
neighbor_results = {}

for person in PEOPLE:
    print(f"\n{'='*60}")
    print(f"Person: {person}")
    print('='*60)

    person_data = neighbor_data.filter(lambda x: person in x['subject'])
    questions = person_data['query']
    answers = person_data['answer']

    if len(questions) == 0:
        print(f"  No questions found for '{person}'.")
        continue

    accuracy = evaluate_neighbours(model, tokenizer, questions, answers, DEVICE)
    neighbor_results[person] = {"accuracy": accuracy, "questions": len(questions)}

In [ ]:
print("=" * 65)
print(f"{'Person':<25} {'Forget':>16} {'Neighbour':>16}")
print("-" * 65)
for person in PEOPLE:
    f = forget_results.get(person)
    n = neighbor_results.get(person)
    f_str = f"{f['accuracy']:.1f}% ({f['questions']}q)" if f else "n/a"
    n_str = f"{n['accuracy']:.1f}% ({n['questions']}q)" if n else "n/a"
    print(f"{person:<25} {f_str:>16} {n_str:>16}")
print("=" * 65)
avg_forget   = sum(d['accuracy'] for d in forget_results.values())   / len(forget_results)
avg_neighbor = sum(d['accuracy'] for d in neighbor_results.values()) / len(neighbor_results)
print(f"{'Average':<25} {avg_forget:>15.1f}% {avg_neighbor:>15.1f}%")